# ☀️ Daylight studio

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/snjsomnath/foton/blob/main/notebooks/daylight_studio.ipynb)

Shape a building, place its windows and a neighbour, and see the daylight factor and annual daylight
autonomy on every floor in about a second. Foton traces the scene on the runtime's GPU; there is no
server and nothing to install on your own machine.

1. **Runtime → Change runtime type → T4 GPU** (the free tier is enough).
2. **Runtime → Run all.** Setup takes about a minute. After that, every slider re-runs the study.

The model is a rectangular Honeybee building with one room and one 0.5 m sensor grid per floor,
sensors 0.75 m above the floor. Windows are 1.5 m high strips at a 0.9 m sill, sized to each façade's
window-to-wall ratio. Annual results use the selected EPW, Honeybee's default 08:00–18:00 occupancy
and a 50 % time target for sDA.

In [ ]:
# @title ⚙️ Setup: installs Foton and connects it to the GPU { display-mode: "form" }
import os, subprocess

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    def sh(cmd):
        subprocess.run(cmd, shell=True, check=True)
    gpu = subprocess.run('nvidia-smi --query-gpu=name --format=csv,noheader', shell=True, capture_output=True, text=True).stdout.strip()
    print('GPU:', gpu or 'none. Use Runtime -> Change runtime type -> T4 GPU, then run again.')
    # Colab has the NVIDIA driver but no Vulkan ICD manifest; point the loader at NVIDIA's headless EGL library.
    sh('apt-get -qq update && apt-get -qq install -y libvulkan1 > /dev/null')
    os.makedirs('/etc/vulkan/icd.d', exist_ok=True)
    with open('/etc/vulkan/icd.d/nvidia_icd.json', 'w') as f:
        f.write('{"file_format_version": "1.0.0", "ICD": {"library_path": "/usr/lib64-nvidia/libEGL_nvidia.so.0", "api_version": "1.3.0"}}')
    sh('pip install -q "foton-daylight[honeybee]>=0.3.5"')
    EPW_DIR = '/content/epw'
    os.makedirs(EPW_DIR, exist_ok=True)
    for city in ['gothenburg', 'seattle', 'phoenix']:
        if not os.path.exists(f'{EPW_DIR}/{city}.epw'):
            sh(f'curl -sfL -o {EPW_DIR}/{city}.epw https://raw.githubusercontent.com/snjsomnath/foton/main/epw/{city}.epw')
else:
    EPW_DIR = os.path.abspath(os.path.join('..', 'epw'))

import math, time
import numpy as np
import foton
from foton.honeybee import HoneybeeStudy
from ladybug_geometry.geometry3d import Point3D
from honeybee.model import Model
from honeybee.room import Room
from honeybee.shade import Shade
from honeybee.facetype import Wall

# One engine for the whole session: it keeps the GPU driver loaded between studies.
ENGINE = foton.Engine({'backend': 'auto'})
DEVICE = ENGINE.capabilities()['name'].split('(')[0].strip()
WEATHER_MEMO = {}
GRID = 0.5   # sensor spacing (m); 0.3 is finer but about 3x the sensors

def cardinal(normal):
    # Compass bearing of an outward wall normal (+Y is north), as N/E/S/W.
    bearing = math.degrees(math.atan2(normal.x, normal.y)) % 360
    return ['north', 'east', 'south', 'west'][int(((bearing + 45) % 360) // 90)]

def build_model(width, depth, floors, storey, rotation, wwr, overhang, fins, neighbour):
    # A rectangular building, one Room and one sensor grid per floor, plus an optional neighbour block.
    rooms, grids = [], []
    for f in range(floors):
        room = Room.from_box(f'Floor_{f + 1}', width, depth, storey, origin=Point3D(-width / 2, -depth / 2, f * storey))
        room.rotate_xy(-rotation, Point3D(0, 0, 0))   # positive rotation turns the building clockwise
        for face in room.faces:
            if isinstance(face.type, Wall) and wwr[cardinal(face.normal)] > 0:
                face.apertures_by_ratio_rectangle(wwr[cardinal(face.normal)], 1.5, 0.9, 2.5)
                for ap in face.apertures:
                    if overhang > 0:
                        ap.overhang(overhang)
                    if fins > 0:
                        ap.left_fin(fins); ap.right_fin(fins)
        rooms.append(room)
        grids.append(room.properties.radiance.generate_sensor_grid(GRID, offset=0.75))
    model = Model('daylight_studio', rooms=rooms)
    model.properties.radiance.sensor_grids = grids
    if neighbour:
        distance, n_floors = neighbour
        half, y0 = max(width, depth) / 2 + 4, -max(width, depth) / 2 - distance
        block = Room.from_box('neighbour', 2 * half, 8, 3.2 * n_floors, origin=Point3D(-half, y0 - 8, 0))
        model.add_shades([Shade(f'neighbour_{i}', face.geometry) for i, face in enumerate(block.faces)])
    return model

def simulate(model, epw, quality='final', annual=True, lux=300):
    t0 = time.perf_counter()
    study = HoneybeeStudy(model, backend='auto', weather_memo=WEATHER_MEMO)
    positions = np.asarray(study.prepared.arrays['sensor_positions'], dtype=np.float32)
    run = study.daylight_factor(quality=quality)
    grids, start = [], 0
    for g in run.grids:
        grids.append({'name': g.room_identifier, 'id': g.full_identifier, 'start': start, 'count': g.sensor_count})
        start += g.sensor_count
    order = np.concatenate([g.sensor_indices for g in run.grids])
    df = np.concatenate([np.asarray(g.values, dtype=np.float32) for g in run.grids])
    da = sda = None
    if annual:
        a = study.annual_daylight(epw, quality=quality, threshold=float(lux), target_time=50.0)
        da = np.concatenate([np.asarray(a.grid(g['id']).da, dtype=np.float32) for g in grids])
        sda = 100.0 * float((da >= 50).mean())
    return {'points': positions[order], 'df': df, 'da': da, 'sda': sda, 'grids': grids,
            'seconds': time.perf_counter() - t0}

print('Foton', foton.__version__ if hasattr(foton, '__version__') else '', 'on', ENGINE.capabilities()['name'])
if not ENGINE.capabilities()['hardware_acceleration']:
    print('Foton is on the CPU: each run will take minutes. Check the runtime type and the Vulkan setup above.')


In [ ]:
# @title ☀️ Daylight studio { display-mode: "form" }
import pandas as pd
import ipywidgets as W
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from IPython.display import display, HTML

if IN_COLAB:
    from google.colab import output as _colab_output
    _colab_output.enable_custom_widget_manager()
    pio.renderers.default = 'colab'

STYLE = '''<style>
.ds-head{background:linear-gradient(115deg,#0b1026 0%,#1d3a8a 55%,#f59e0b 120%);color:#fff;border-radius:16px;
  padding:20px 24px;font-family:Inter,system-ui,sans-serif;margin-bottom:10px}
.ds-head h1{margin:0;font-size:28px;font-weight:700;letter-spacing:-.02em;color:#fff}
.ds-head p{margin:4px 0 0;opacity:.85;font-size:14px}
.ds-badge{display:inline-block;margin-top:10px;padding:3px 11px;border-radius:999px;background:rgba(255,255,255,.16);font-size:12px}
.ds-cards{display:flex;gap:10px;flex-wrap:wrap;font-family:Inter,system-ui,sans-serif;margin:4px 0 8px}
.ds-card{flex:1;min-width:118px;background:#0b1026;color:#cbd5e1;border-radius:12px;padding:11px 14px}
.ds-card b{display:block;font-size:24px;color:#fbbf24;font-weight:700}
.ds-card span{font-size:12px}
.ds-note{font-family:Inter,system-ui,sans-serif;font-size:13px;color:#64748b}
.ds-err{font-family:Inter,system-ui,sans-serif;color:#b91c1c;background:#fef2f2;border-radius:10px;padding:10px 14px}
</style>'''
SCALE = [[0, '#0b1026'], [0.2, '#1d3a8a'], [0.45, '#0e9f9a'], [0.7, '#f5d64a'], [1, '#fff7d6']]

def cards(items):
    return HTML('<div class="ds-cards">' + ''.join(f'<div class="ds-card"><b>{v}</b><span>{k}</span></div>' for k, v in items) + '</div>')

# ── Controls ──────────────────────────────────────────────────────────────────
_w = {'style': {'description_width': '118px'}, 'layout': W.Layout(width='330px'), 'continuous_update': False}
def fs(desc, value, lo, hi, step, fmt='.1f'):
    return W.FloatSlider(value=value, min=lo, max=hi, step=step, description=desc, readout_format=fmt, **_w)
def ints(desc, value, lo, hi):
    return W.IntSlider(value=value, min=lo, max=hi, description=desc, **_w)

width, depth = fs('Width (m)', 10, 3, 24, 0.5), fs('Depth (m)', 12, 3, 18, 0.5)
floors, storey = ints('Floors', 3, 1, 6), fs('Floor height (m)', 3.2, 2.6, 4.5, 0.1)
rotation = fs('Rotation (°)', 0, -180, 180, 5, '.0f')
wwr = {d: fs(f'WWR {d}', v, 0, 0.9, 0.05, '.2f') for d, v in [('north', 0.25), ('east', 0.15), ('south', 0.35), ('west', 0.15)]}
overhang, fins = fs('Overhang (m)', 0, 0, 2, 0.1), fs('Side fins (m)', 0, 0, 1.5, 0.1)
nb_on = W.Checkbox(value=True, description='Neighbour to the south', indent=False)
nb_dist, nb_floors = fs('Distance (m)', 8, 2, 30, 0.5), ints('Neighbour floors', 5, 1, 12)
climate = W.Dropdown(options=[('Gothenburg', 'gothenburg'), ('Seattle', 'seattle'), ('Phoenix', 'phoenix')], value='gothenburg',
                     description='Climate', style=_w['style'], layout=_w['layout'])
quality = W.ToggleButtons(options=[('Final', 'final'), ('Fast (noisier)', 'preview')], value='final', style={'button_width': '110px'})
annual = W.Checkbox(value=True, description='Annual daylight (DA / sDA)', indent=False)
lux = W.IntSlider(value=300, min=100, max=1000, step=50, description='DA threshold (lx)', **_w)
live = W.Checkbox(value=True, description='Re-run when a slider moves', indent=False)
colour_by = W.ToggleButtons(options=[('Daylight factor', 'df'), ('Daylight autonomy', 'da')], value='df',
                            style={'button_width': '140px'})
run_btn = W.Button(description='Simulate', icon='sun-o', button_style='warning', layout=W.Layout(width='160px', height='38px'))
status = W.HTML()

panel = W.Accordion(children=[
    W.VBox([width, depth, floors, storey, rotation]),
    W.VBox(list(wwr.values()) + [overhang, fins]),
    W.VBox([nb_on, nb_dist, nb_floors]),
    W.VBox([climate, quality, annual, lux, live]),
])
for i, t in enumerate(['🏢 Building', '🪟 Windows & shading', '🏙️ Neighbour', '☀️ Analysis']):
    panel.set_title(i, t)
panel.selected_index = 0

out_cards, out_view, out_plan, out_hist = W.Output(), W.Output(), W.Output(), W.Output()
tabs = W.Tab(children=[out_view, out_plan, out_hist])
for i, t in enumerate(['3D view', 'Floor plans', 'History']):
    tabs.set_title(i, t)

# ── Figures ───────────────────────────────────────────────────────────────────
def _mesh(polys, color, opacity):
    xs, ys, zs, I, J, K = [], [], [], [], [], []
    for verts in polys:
        n0 = len(xs)
        for p in verts:
            xs.append(p.x); ys.append(p.y); zs.append(p.z)
        for t in range(1, len(verts) - 1):
            I.append(n0); J.append(n0 + t); K.append(n0 + t + 1)
    return go.Mesh3d(x=xs, y=ys, z=zs, i=I, j=J, k=K, color=color, opacity=opacity,
                     hoverinfo='skip', flatshading=True, showlegend=False)

def _edges(polys, color):
    xs, ys, zs = [], [], []
    for verts in polys:
        for p in list(verts) + [verts[0]]:
            xs.append(p.x); ys.append(p.y); zs.append(p.z)
        xs.append(None); ys.append(None); zs.append(None)
    return go.Scatter3d(x=xs, y=ys, z=zs, mode='lines', line=dict(color=color, width=2), hoverinfo='skip', showlegend=False)

def view_3d(model, pts, values, label):
    walls, glass, shades = [], [], []
    for room in model.rooms:
        for face in room.faces:
            walls.append(face.geometry.vertices)
            for ap in face.apertures:
                glass.append(ap.geometry.vertices)
                shades += [s.geometry.vertices for s in ap.outdoor_shades]
    context = [s.geometry.vertices for s in model.orphaned_shades]
    fig = go.Figure([
        _mesh(walls, '#e2e8f0', 0.12), _edges(walls, '#94a3b8'),
        _mesh(glass, '#38bdf8', 0.55), _mesh(shades, '#475569', 0.8),
        _mesh(context, '#cbd5e1', 0.14), _edges(context, '#cbd5e1'),
        go.Scatter3d(x=pts[:, 0], y=pts[:, 1], z=pts[:, 2], mode='markers', showlegend=False,
                     marker=dict(size=3.2, color=values, colorscale=SCALE, cmin=0, cmax=max(float(values.max()), 1e-6),
                                 colorbar=dict(title=label, thickness=14, len=0.7)),
                     hovertemplate=label + ' %{marker.color:.2f}<extra></extra>'),
    ])
    span = max(width.value, depth.value, 10)
    fig.add_trace(go.Scatter3d(x=[0, 0], y=[span * 0.75, span * 1.05], z=[0, 0], mode='lines+text', text=['', 'N'],
                               line=dict(color='#f59e0b', width=6), textfont=dict(color='#f59e0b', size=16),
                               showlegend=False, hoverinfo='skip'))
    fig.update_layout(height=560, margin=dict(l=0, r=0, t=10, b=0), paper_bgcolor='rgba(0,0,0,0)',
                      scene=dict(aspectmode='data', xaxis_visible=False, yaxis_visible=False, zaxis_visible=False,
                                 camera=dict(eye=dict(x=0.9, y=-1.2, z=1.25))))
    return fig

def floor_plans(result, values, label):
    pts, grids = result['points'], result['grids']
    fig = make_subplots(rows=1, cols=len(grids), subplot_titles=[g['name'].replace('_', ' ') for g in grids],
                        horizontal_spacing=0.04)
    vmax = max(float(values.max()), 1e-6)
    for c, g in enumerate(grids, start=1):
        s = slice(g['start'], g['start'] + g['count'])
        fig.add_trace(go.Scattergl(x=pts[s, 0], y=pts[s, 1], mode='markers', showlegend=False,
                                   marker=dict(symbol='square', size=11, color=values[s], colorscale=SCALE, cmin=0, cmax=vmax,
                                               colorbar=dict(title=label, thickness=14) if c == 1 else None, showscale=c == 1),
                                   hovertemplate=label + ' %{marker.color:.2f}<extra></extra>'), row=1, col=c)
        fig.update_xaxes(visible=False, row=1, col=c)
        fig.update_yaxes(visible=False, scaleanchor=f'x{c if c > 1 else ""}', row=1, col=c)
    fig.update_layout(height=380, margin=dict(l=0, r=0, t=30, b=0), paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)')
    return fig

# ── Actions ───────────────────────────────────────────────────────────────────
state = {'busy': False, 'last': None, 'history': []}

def current_model():
    return build_model(width.value, depth.value, floors.value, storey.value, rotation.value,
                       {k: s.value for k, s in wwr.items()}, overhang.value, fins.value,
                       (nb_dist.value, nb_floors.value) if nb_on.value else None)

def draw():
    if not state['last']:
        return
    result, model = state['last']
    key = colour_by.value if result['da'] is not None else 'df'
    values, label = result[key], 'DF %' if key == 'df' else 'DA %'
    with out_view:
        out_view.clear_output(wait=True)
        view_3d(model, result['points'], values, label).show()
    with out_plan:
        out_plan.clear_output(wait=True)
        display(colour_by)
        floor_plans(result, values, label).show()

def run(_=None):
    if state['busy']:
        return
    state['busy'], run_btn.disabled = True, True
    status.value = '<span class="ds-note">⏳ Simulating…</span>'
    try:
        model = current_model()
        result = simulate(model, f'{EPW_DIR}/{climate.value}.epw', quality.value, annual.value, lux.value)
        state['last'] = (result, model)
        df, sda, seconds = result['df'], result['sda'], result['seconds']
        state['history'].append({
            'climate': climate.label, 'width': width.value, 'depth': depth.value, 'floors': floors.value,
            'rotation': rotation.value, **{f'wwr_{k}': s.value for k, s in wwr.items()},
            'overhang': overhang.value, 'fins': fins.value, 'neighbour': nb_floors.value if nb_on.value else 0,
            'quality': quality.value, 'mean DF %': round(float(df.mean()), 2), 'area ≥2% DF': f'{(df >= 2).mean():.0%}',
            f'sDA{lux.value}/50 %': None if sda is None else round(sda, 1), 'seconds': round(seconds, 2)})
        with out_cards:
            out_cards.clear_output(wait=True)
            display(cards([('mean daylight factor', f'{df.mean():.1f} %'), ('floor area ≥ 2 % DF', f'{(df >= 2).mean():.0%}'),
                           (f'sDA {lux.value} lx / 50 %', '—' if sda is None else f'{sda:.0f} %'),
                           ('sensors', f'{df.size:,}'), ('compute time', f'{seconds:.2f} s')]))
        with out_hist:
            out_hist.clear_output(wait=True)
            display(pd.DataFrame(state['history'][::-1]))
        draw()
        status.value = f'<span class="ds-note">✅ {seconds:.2f} s on {DEVICE}</span>'
    except Exception as exc:
        status.value = f'<div class="ds-err">⚠️ {type(exc).__name__}: {str(exc)[:300]}</div>'
    finally:
        state['busy'], run_btn.disabled = False, False

run_btn.on_click(run)
colour_by.observe(lambda _: draw(), names='value')
for ctl in [width, depth, floors, storey, rotation, overhang, fins, nb_on, nb_dist, nb_floors, climate, quality, annual, lux, *wwr.values()]:
    ctl.observe(lambda _: live.value and run(), names='value')

display(HTML(STYLE + f'''<div class="ds-head"><h1>☀️ Daylight studio</h1>
<p>Shape a building, place its windows and neighbours, and see daylight on every floor in seconds.</p>
<span class="ds-badge">⚡ Foton on {DEVICE}</span></div>'''))
display(W.HBox([W.VBox([panel, run_btn, status], layout=W.Layout(width='350px', min_width='350px')),
                W.VBox([out_cards, tabs], layout=W.Layout(flex='1', min_width='0'))]))
run()


## Diagnostics
Optional. Checks that the numbers behave (no windows means no daylight, a close neighbour lowers it) and measures how many studies a minute this GPU runs.

In [ ]:
# Runs without the studio: checks that results are sane and measures throughput on this GPU.
checks = []
def check(name, ok, detail=''):
    checks.append((name, bool(ok), detail))

base = dict(width=8, depth=6, floors=2, storey=3.2, rotation=0, wwr={'north': 0.4, 'east': 0.3, 'south': 0.3, 'west': 0.3},
            overhang=0, fins=0, neighbour=None)
r = simulate(build_model(**base), f'{EPW_DIR}/gothenburg.epw', 'final', True)
check('daylight factor in range', 0 < r['df'].min() <= r['df'].max() < 100, f"mean {r['df'].mean():.2f} %, {r['df'].size} sensors")
check('one grid per floor', [g['name'] for g in r['grids']] == ['Floor_1', 'Floor_2'])
check('annual DA in range', r['da'] is not None and 0 <= r['da'].min() <= r['da'].max() <= 100, f"sDA {r['sda']:.0f} %")
blank = simulate(build_model(**{**base, 'wwr': {k: 0.0 for k in base['wwr']}}), f'{EPW_DIR}/gothenburg.epw', 'preview', False)
check('no windows gives no daylight', blank['df'].max() < 0.01, f"max DF {blank['df'].max():.3f} %")
shaded = simulate(build_model(**{**base, 'neighbour': (3, 10)}), f'{EPW_DIR}/gothenburg.epw', 'final', False)
check('a close neighbour lowers daylight', shaded['df'].mean() < r['df'].mean(), f"{r['df'].mean():.2f} -> {shaded['df'].mean():.2f} %")

rng = np.random.default_rng(0)
times = []
for _ in range(20):
    m = build_model(float(rng.uniform(5, 14)), float(rng.uniform(4, 12)), int(rng.integers(1, 4)), 3.2, 0,
                    {k: float(rng.uniform(0.2, 0.6)) for k in base['wwr']}, 0, 0, None)
    times.append(simulate(m, f'{EPW_DIR}/gothenburg.epw', 'preview', False)['seconds'])
check('throughput, 20 buildings', True, f'{60 / np.mean(times):.0f} studies/min, median {np.median(times):.2f} s')

print(DEVICE, '\n')
for name, ok, detail in checks:
    print(f"{'PASS' if ok else 'FAIL':4}  {name:34} {detail}")
